# EDA Inicial del Dataset de Seguridad en Obras de Construcción

Este notebook realiza un análisis exploratorio inicial del dataset de imágenes `snehilsanyal/construction-site-safety-image-dataset-roboflow`, orientado a detección de uso de EPP/PPE en obras de construcción.

**Alcance:** inspeccionar, analizar, visualizar y reportar el estado inicial del dataset.

**Restricción importante:** no se realiza limpieza, eliminación, movimiento, redimensionamiento, transformación de imágenes ni entrenamiento de modelos.

## 0. Configuración inicial

Se importan las librerías necesarias y se definen constantes reutilizables. El análisis usa `pathlib` para rutas, `PIL/Pillow` para validar imágenes, `pandas` y `numpy` para análisis tabular, `matplotlib` para visualizaciones, `hashlib` para duplicados exactos y `opencv-python` solo para estimar desenfoque.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import random
import warnings

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageStat, UnidentifiedImageError
from tqdm.auto import tqdm

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LABEL_EXTENSIONS = {".txt", ".xml", ".json", ".csv", ".yaml", ".yml"}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

plt.style.use("default")

## 1. Descarga y ruta raíz del dataset

La siguiente celda descarga la versión más reciente disponible mediante `kagglehub`. La ruta devuelta se usa como raíz para todo el análisis.

In [ ]:
# Download latest version
DATASET_HANDLE = "snehilsanyal/construction-site-safety-image-dataset-roboflow"
path = kagglehub.dataset_download(DATASET_HANDLE)
DOWNLOAD_ROOT = Path(path)
DATASET_ROOT = DOWNLOAD_ROOT / "css-data"

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"No se encontro la carpeta esperada del dataset YOLO: {DATASET_ROOT}")

print("Path to downloaded files:", DOWNLOAD_ROOT)
print("Dataset root used for EDA:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())

## 2. Funciones auxiliares

Estas funciones inspeccionan el dataset sin modificarlo. Están separadas para mantener el notebook legible y facilitar su reutilización.

In [ ]:
def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)


def detect_split(path: Path, root: Path) -> str:
    parts = [p.lower() for p in path.relative_to(root).parts]
    if "train" in parts:
        return "train"
    if "valid" in parts or "validation" in parts or "val" in parts:
        return "validation"
    if "test" in parts:
        return "test"
    return "unknown"


def find_files(root: Path, extensions: set[str]) -> list[Path]:
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in extensions])


def print_tree(root: Path, max_depth: int = 3, max_items_per_dir: int = 12) -> None:
    root = root.resolve()
    print(root)
    for directory in sorted([p for p in root.rglob("*") if p.is_dir()]):
        rel = directory.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * depth
        children = sorted(directory.iterdir())[:max_items_per_dir]
        print(f"{indent}{directory.name}/")
        for child in children:
            if child.is_file():
                print(f"{indent}  {child.name}")


def md5_hash(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = hashlib.md5()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def yolo_image_path_for_label(label_path: Path, image_paths_by_stem: dict[tuple[str, str], list[Path]]) -> Path | None:
    split = detect_split(label_path, DATASET_ROOT)
    key = (split, label_path.stem)
    candidates = image_paths_by_stem.get(key, [])
    if candidates:
        return candidates[0]
    # fallback when split cannot be inferred cleanly
    all_candidates = []
    for (candidate_split, stem), paths in image_paths_by_stem.items():
        if stem == label_path.stem:
            all_candidates.extend(paths)
    return all_candidates[0] if all_candidates else None

## 3. Estructura del dataset

Se listan carpetas, subcarpetas y archivos principales para identificar cómo están organizadas las imágenes y las etiquetas, y si existen divisiones `train`, `validation` y `test`.

In [ ]:
print_tree(DATASET_ROOT, max_depth=3, max_items_per_dir=10)

all_files = sorted([p for p in DATASET_ROOT.rglob("*") if p.is_file()])
image_paths = find_files(DATASET_ROOT, IMAGE_EXTENSIONS)
label_paths = find_files(DATASET_ROOT, LABEL_EXTENSIONS)

structure_summary = pd.DataFrame({
    "metric": ["total_files", "image_files", "label_or_metadata_files", "directories"],
    "value": [
        len(all_files),
        len(image_paths),
        len(label_paths),
        len([p for p in DATASET_ROOT.rglob("*") if p.is_dir()]),
    ],
})
structure_summary

In [ ]:
split_counts_files = pd.Series([detect_split(p, DATASET_ROOT) for p in image_paths], name="split").value_counts().rename_axis("split").reset_index(name="image_count")
split_counts_files

## 4. Inventario de imágenes

Se crea un inventario tabular con ruta, split, extensión y tamaño de archivo. Esto permite responder cuántas imágenes existen, en qué formatos están y si hay archivos atípicamente pequeños o grandes.

In [ ]:
image_inventory = pd.DataFrame([
    {
        "path": str(p),
        "relative_path": str(p.relative_to(DATASET_ROOT)),
        "file_name": p.name,
        "stem": p.stem,
        "extension": p.suffix.lower(),
        "split": detect_split(p, DATASET_ROOT),
        "size_bytes": p.stat().st_size,
        "size_kb": p.stat().st_size / 1024,
        "size_mb": file_size_mb(p),
    }
    for p in image_paths
])

print(f"Total de imágenes: {len(image_inventory):,}")
display(image_inventory.head())

In [ ]:
display(image_inventory.groupby("split").size().rename("images").reset_index())
display(image_inventory.groupby("extension").size().rename("images").reset_index().sort_values("images", ascending=False))

display(image_inventory["size_kb"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("size_kb"))

fig, ax = plt.subplots(figsize=(9, 4))
image_inventory["size_kb"].hist(bins=50, ax=ax)
ax.set_title("Distribución del tamaño de archivos de imagen")
ax.set_xlabel("Tamaño (KB)")
ax.set_ylabel("Número de imágenes")
plt.show()

In [ ]:
size_q1 = image_inventory["size_kb"].quantile(0.25)
size_q3 = image_inventory["size_kb"].quantile(0.75)
size_iqr = size_q3 - size_q1
small_threshold = max(0, size_q1 - 1.5 * size_iqr)
large_threshold = size_q3 + 1.5 * size_iqr

size_outliers = image_inventory[(image_inventory["size_kb"] < small_threshold) | (image_inventory["size_kb"] > large_threshold)].copy()
print("Umbral pequeño (KB):", round(small_threshold, 2))
print("Umbral grande (KB):", round(large_threshold, 2))
print("Posibles outliers por tamaño:", len(size_outliers))
display(size_outliers.sort_values("size_kb").head(10))
display(size_outliers.sort_values("size_kb", ascending=False).head(10))

## 5. Clases y etiquetas

Los datasets exportados desde Roboflow suelen estar en formato YOLO, con archivos `.txt` por imagen y un `data.yaml` con nombres de clases. Esta sección intenta leer esa estructura sin asumir que siempre exista.

In [ ]:
yaml_files = sorted(DATASET_ROOT.rglob("*.yaml")) + sorted(DATASET_ROOT.rglob("*.yml"))
print("Archivos YAML encontrados:")
for yaml_file in yaml_files:
    print("-", yaml_file.relative_to(DATASET_ROOT))

In [ ]:
import re

def parse_yolo_class_names_from_yaml(yaml_path: Path) -> dict[int, str]:
    """Parser simple para la sección names de data.yaml sin depender de PyYAML."""
    content = yaml_path.read_text(encoding="utf-8", errors="ignore")
    
    # Intentar encontrar arreglo de nombres en una sola línea (ej. names: [A, B] o {names: [A, B]})
    match = re.search(r'names:\s*\[(.*?)\]', content, re.DOTALL)
    if match:
        items = match.group(1).split(',')
        return {idx: item.strip().strip("'\"") for idx, item in enumerate(items) if item.strip()}

    # Fallback a lectura línea por línea
    names = {}
    in_names = False
    for line in content.splitlines():
        stripped = line.strip()
        if stripped.startswith("names:"):
            in_names = True
            continue
        if in_names:
            if not stripped or stripped.startswith("#"):
                continue
            if not line.startswith(" ") and not line.startswith("-"):
                break
            if ":" in stripped:
                key, value = stripped.split(":", 1)
                if key.strip().isdigit():
                    names[int(key.strip())] = value.strip().strip("'\"")
            elif stripped.startswith("-"):
                names[len(names)] = stripped[1:].strip().strip("'\"")
                
    return names

class_names = {}
for yaml_file in yaml_files:
    parsed = parse_yolo_class_names_from_yaml(yaml_file)
    if parsed:
        class_names = parsed
        print(f"Clases leídas desde: {yaml_file.relative_to(DATASET_ROOT)}")
        break

print("Número de clases:", len(class_names))
class_names_df = pd.DataFrame([{"class_id": k, "class_name": v} for k, v in sorted(class_names.items())])
class_names_df


In [ ]:
yolo_label_paths = [p for p in DATASET_ROOT.rglob("*.txt") if "label" in str(p.parent).lower() or p.parent.name.lower() == "labels"]
print("Archivos de etiquetas YOLO candidatos:", len(yolo_label_paths))

yolo_rows = []
for label_path in tqdm(yolo_label_paths, desc="Leyendo etiquetas YOLO"):
    split = detect_split(label_path, DATASET_ROOT)
    text = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    for line_number, line in enumerate(text, start=1):
        parts = line.strip().split()
        if len(parts) >= 5 and parts[0].lstrip("-").isdigit():
            class_id = int(parts[0])
            yolo_rows.append({
                "label_path": str(label_path),
                "relative_label_path": str(label_path.relative_to(DATASET_ROOT)),
                "label_stem": label_path.stem,
                "split": split,
                "line_number": line_number,
                "class_id": class_id,
                "class_name": class_names.get(class_id, f"class_{class_id}"),
                "x_center": float(parts[1]),
                "y_center": float(parts[2]),
                "width": float(parts[3]),
                "height": float(parts[4]),
            })

annotations_df = pd.DataFrame(yolo_rows)
print("Anotaciones encontradas:", len(annotations_df))
display(annotations_df.head())

In [ ]:
if not annotations_df.empty:
    annotations_by_class = annotations_df.groupby(["class_id", "class_name"]).size().reset_index(name="annotation_count").sort_values("annotation_count", ascending=False)
    images_by_class = annotations_df.groupby(["class_id", "class_name"])["label_stem"].nunique().reset_index(name="image_count").sort_values("image_count", ascending=False)
    display(annotations_by_class)
    display(images_by_class)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(annotations_by_class))))
    ax.barh(annotations_by_class["class_name"], annotations_by_class["annotation_count"])
    ax.set_title("Anotaciones por clase")
    ax.set_xlabel("Número de anotaciones")
    ax.invert_yaxis()
    plt.show()
else:
    print("No se encontraron anotaciones YOLO parseables.")

In [ ]:
if not annotations_df.empty:
    class_balance = annotations_df["class_name"].value_counts()
    max_count = class_balance.max()
    min_count = class_balance.min()
    imbalance_ratio = max_count / min_count if min_count else np.nan
    print("Clase más frecuente:", class_balance.idxmax(), int(max_count))
    print("Clase menos frecuente:", class_balance.idxmin(), int(min_count))
    print("Ratio max/min:", round(float(imbalance_ratio), 2))

    if not class_names_df.empty:
        ppe_absence_keywords = ["no", "without", "missing", "sin", "no-", "no_", "no ", "nohelmet", "nomask", "novest"]
        absence_classes = class_names_df[class_names_df["class_name"].str.lower().apply(lambda name: any(keyword in name for keyword in ppe_absence_keywords))]
        print("Clases potencialmente relacionadas con ausencia de EPP:")
        display(absence_classes)
    else:
        print("No se encontraron nombres de clases para analizar la ausencia de EPP.")


## 6. Dimensiones y aspect ratio de imágenes

Se intenta abrir cada imagen con Pillow para obtener ancho, alto, modo de color y métricas básicas. Las imágenes corruptas o ilegibles se reportan, no se eliminan.

In [ ]:
image_metadata_rows = []
invalid_images = []

for image_path in tqdm(image_paths, desc="Inspeccionando imágenes"):
    row = {
        "path": str(image_path),
        "relative_path": str(image_path.relative_to(DATASET_ROOT)),
        "file_name": image_path.name,
        "stem": image_path.stem,
        "split": detect_split(image_path, DATASET_ROOT),
        "extension": image_path.suffix.lower(),
        "size_kb": image_path.stat().st_size / 1024,
        "is_valid": False,
        "width": np.nan,
        "height": np.nan,
        "aspect_ratio": np.nan,
        "mode": None,
        "brightness": np.nan,
        "contrast": np.nan,
        "laplacian_variance": np.nan,
    }
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            row["width"], row["height"] = img.size
            row["aspect_ratio"] = row["width"] / row["height"] if row["height"] else np.nan
            row["mode"] = img.mode
            gray = img.convert("L")
            stat = ImageStat.Stat(gray)
            row["brightness"] = stat.mean[0]
            row["contrast"] = stat.stddev[0]
            if CV2_AVAILABLE:
                gray_np = np.array(gray)
                row["laplacian_variance"] = cv2.Laplacian(gray_np, cv2.CV_64F).var()
            row["is_valid"] = True
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        row["error"] = repr(exc)
        invalid_images.append(row)
    image_metadata_rows.append(row)

image_metadata = pd.DataFrame(image_metadata_rows)
invalid_images_df = pd.DataFrame(invalid_images)

print("Imágenes válidas:", int(image_metadata["is_valid"].sum()))
print("Imágenes ilegibles o corruptas:", len(invalid_images_df))
display(invalid_images_df.head())

In [ ]:
valid_images_df = image_metadata[image_metadata["is_valid"]].copy()

display(valid_images_df[["width", "height", "aspect_ratio"]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
display(valid_images_df.groupby("mode").size().rename("image_count").reset_index().sort_values("image_count", ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
valid_images_df["width"].hist(bins=40, ax=axes[0])
axes[0].set_title("Distribución de ancho")
axes[0].set_xlabel("Pixeles")
valid_images_df["height"].hist(bins=40, ax=axes[1])
axes[1].set_title("Distribución de alto")
axes[1].set_xlabel("Pixeles")
valid_images_df["aspect_ratio"].hist(bins=40, ax=axes[2])
axes[2].set_title("Distribución de aspect ratio")
axes[2].set_xlabel("Ancho / alto")
plt.tight_layout()
plt.show()

In [ ]:
if not valid_images_df.empty:
    small_dimension_outliers = valid_images_df[(valid_images_df["width"] < 100) | (valid_images_df["height"] < 100)]
    unusual_ratio_outliers = valid_images_df[(valid_images_df["aspect_ratio"] < 0.5) | (valid_images_df["aspect_ratio"] > 2.0)]
    print("Imágenes con dimensión menor a 100 px:", len(small_dimension_outliers))
    display(small_dimension_outliers[["relative_path", "width", "height", "aspect_ratio"]].head(20))
    print("Imágenes con proporciones potencialmente inusuales (<0.5 o >2.0):", len(unusual_ratio_outliers))
    display(unusual_ratio_outliers[["relative_path", "width", "height", "aspect_ratio"]].head(20))

## 7. Análisis de duplicados exactos

Se calculan hashes MD5 por archivo para detectar duplicados exactos. Este análisis solo reporta grupos duplicados; no elimina ni modifica archivos.

In [ ]:
hash_rows = []
for image_path in tqdm(image_paths, desc="Calculando hashes MD5"):
    try:
        hash_rows.append({
            "path": str(image_path),
            "relative_path": str(image_path.relative_to(DATASET_ROOT)),
            "split": detect_split(image_path, DATASET_ROOT),
            "file_name": image_path.name,
            "md5": md5_hash(image_path),
            "size_kb": image_path.stat().st_size / 1024,
        })
    except OSError as exc:
        hash_rows.append({
            "path": str(image_path),
            "relative_path": str(image_path.relative_to(DATASET_ROOT)),
            "split": detect_split(image_path, DATASET_ROOT),
            "file_name": image_path.name,
            "md5": None,
            "size_kb": image_path.stat().st_size / 1024,
            "hash_error": repr(exc),
        })

hash_df = pd.DataFrame(hash_rows)
duplicate_hashes = hash_df[hash_df["md5"].notna()].groupby("md5").filter(lambda group: len(group) > 1)
print("Imágenes en grupos duplicados exactos:", len(duplicate_hashes))
print("Número de grupos duplicados:", duplicate_hashes["md5"].nunique())
display(duplicate_hashes.sort_values(["md5", "relative_path"]).head(50))

## 8. Inspección visual

Se muestran muestras aleatorias, ejemplos por split y ejemplos por clase cuando las etiquetas lo permiten. Las cajas se dibujan solo sobre copias en memoria para visualización; los archivos originales no se modifican.

In [ ]:
def show_image_grid(paths, title, max_images=12, cols=4):
    paths = list(paths)[:max_images]
    if not paths:
        print(f"Sin imágenes para mostrar: {title}")
        return
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, image_path in zip(axes, paths):
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(f"{detect_split(image_path, DATASET_ROOT)} | {image_path.name}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(paths):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

sample_paths = random.sample(image_paths, k=min(12, len(image_paths)))
show_image_grid(sample_paths, "Muestra aleatoria de imágenes")

In [ ]:
for split_name in ["train", "validation", "test", "unknown"]:
    split_paths = [p for p in image_paths if detect_split(p, DATASET_ROOT) == split_name]
    if split_paths:
        show_image_grid(random.sample(split_paths, k=min(8, len(split_paths))), f"Ejemplos del split: {split_name}", max_images=8, cols=4)

In [ ]:
def draw_yolo_annotations(image_path: Path, annotations: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(7, 7))
    with Image.open(image_path) as img:
        img_rgb = img.convert("RGB")
        width, height = img_rgb.size
        ax.imshow(img_rgb)
    for _, ann in annotations.iterrows():
        x_center = ann["x_center"] * width
        y_center = ann["y_center"] * height
        box_width = ann["width"] * width
        box_height = ann["height"] * height
        x0 = x_center - box_width / 2
        y0 = y_center - box_height / 2
        rect = plt.Rectangle((x0, y0), box_width, box_height, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x0, max(0, y0 - 3), ann["class_name"], fontsize=9, color="white", bbox={"facecolor": "black", "alpha": 0.6, "pad": 1})
    ax.set_title(image_path.name)
    ax.axis("off")
    plt.show()

if not annotations_df.empty:
    image_paths_by_stem = defaultdict(list)
    for image_path in image_paths:
        image_paths_by_stem[(detect_split(image_path, DATASET_ROOT), image_path.stem)].append(image_path)

    for class_name in annotations_df["class_name"].drop_duplicates().head(8):
        class_sample = annotations_df[annotations_df["class_name"] == class_name].sample(1, random_state=RANDOM_SEED).iloc[0]
        label_path = Path(class_sample["label_path"])
        image_path = yolo_image_path_for_label(label_path, image_paths_by_stem)
        if image_path is not None:
            image_annotations = annotations_df[(annotations_df["label_path"] == str(label_path))]
            print(f"Ejemplo para clase: {class_name}")
            draw_yolo_annotations(image_path, image_annotations)
else:
    print("No hay anotaciones disponibles para mostrar ejemplos por clase.")

## 9. Indicadores básicos de calidad de imagen

Se estiman brillo, contraste y desenfoque usando la varianza del Laplaciano cuando OpenCV está disponible. Estos indicadores son aproximaciones iniciales para detectar posibles outliers visuales.

In [ ]:
quality_columns = ["brightness", "contrast"] + (["laplacian_variance"] if CV2_AVAILABLE else [])
display(valid_images_df[quality_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

fig, axes = plt.subplots(1, len(quality_columns), figsize=(5 * len(quality_columns), 4))
if len(quality_columns) == 1:
    axes = [axes]
for ax, column in zip(axes, quality_columns):
    valid_images_df[column].dropna().hist(bins=50, ax=ax)
    ax.set_title(f"Distribución de {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Número de imágenes")
plt.tight_layout()
plt.show()

In [ ]:
def iqr_outliers(df: pd.DataFrame, column: str) -> pd.DataFrame:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[(df[column] < lower) | (df[column] > upper)].copy()

for column in quality_columns:
    outliers = iqr_outliers(valid_images_df, column)
    print(f"Outliers potenciales para {column}: {len(outliers)}")
    display(outliers[["relative_path", "split", column, "width", "height", "mode"]].sort_values(column).head(5))
    display(outliers[["relative_path", "split", column, "width", "height", "mode"]].sort_values(column, ascending=False).head(5))

## 10. Modelos YOLO utilizados y comparación simple

Esta sección resume los dos modelos de detección usados en el proyecto: `YOLOv8n` como línea base y `YOLOv8s` como versión final. No se entrena nada aquí; solo se visualizan métricas ya reportadas en el notebook de entrenamiento y evaluación para dejar la comparación dentro del EDA.


In [ ]:
yolo_model_metrics = pd.DataFrame([
    {
        "modelo": "YOLOv8n línea base",
        "version": "yolov8n",
        "run": "ppe_yolov8n_baseline",
        "precision": 0.861,
        "recall": 0.662,
        "mAP50": 0.721,
        "mAP50_95": 0.409,
    },
    {
        "modelo": "YOLOv8s final",
        "version": "yolov8s",
        "run": "ppe_yolov8s_recall_focus",
        "precision": 0.898,
        "recall": 0.737,
        "mAP50": 0.784,
        "mAP50_95": 0.508,
    },
])

metric_columns = ["precision", "recall", "mAP50", "mAP50_95"]
display(yolo_model_metrics)

plot_df = yolo_model_metrics.set_index("modelo")[metric_columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_df.plot(kind="bar", ax=axes[0])
axes[0].set_title("Comparación general de métricas YOLO")
axes[0].set_xlabel("Modelo")
axes[0].set_ylabel("Valor de la métrica")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Métrica", loc="lower right")

improvement = (
    plot_df.loc["YOLOv8s final"] - plot_df.loc["YOLOv8n línea base"]
).rename("mejora") * 100
improvement.plot(kind="bar", ax=axes[1], color="#4C78A8")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Mejora de YOLOv8s frente a YOLOv8n")
axes[1].set_xlabel("Métrica")
axes[1].set_ylabel("Diferencia en puntos porcentuales")
axes[1].tick_params(axis="x", rotation=0)
for index, value in enumerate(improvement):
    axes[1].text(index, value + 0.5, f"{value:.1f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()


La comparación se enfoca en `recall` y `mAP50-95` porque el caso de uso es seguridad: omitir una infracción puede ser más costoso que mandar un caso dudoso a revisión humana.


In [ ]:
critical_class_recall = pd.DataFrame([
    {"clase": "NO-Hardhat", "YOLOv8n línea base": 0.537, "YOLOv8s final": 0.576},
    {"clase": "NO-Mask", "YOLOv8n línea base": 0.587, "YOLOv8s final": 0.732},
    {"clase": "NO-Safety Vest", "YOLOv8n línea base": 0.722, "YOLOv8s final": 0.733},
])

display(critical_class_recall)

fig, ax = plt.subplots(figsize=(9, 4))
critical_class_recall.set_index("clase").plot(kind="bar", ax=ax)
ax.set_title("Recall en clases críticas de incumplimiento")
ax.set_xlabel("Clase")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Modelo")
plt.tight_layout()
plt.show()

critical_class_recall["mejora_recall_pp"] = (
    critical_class_recall["YOLOv8s final"] - critical_class_recall["YOLOv8n línea base"]
) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(critical_class_recall["clase"], critical_class_recall["mejora_recall_pp"], color="#59A14F")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Cambio de recall por clase crítica")
ax.set_xlabel("Clase")
ax.set_ylabel("Mejora en puntos porcentuales")
for index, value in enumerate(critical_class_recall["mejora_recall_pp"]):
    ax.text(index, value + 0.4, f"{value:.1f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


Lectura rápida: `YOLOv8s` mejora todas las métricas globales y sube el recall de las tres clases `NO-*`. La mejora más clara está en `NO-Mask`; `NO-Hardhat` sigue siendo la clase más débil y debería revisarse en análisis posteriores.


## 11. Resumen final

Ejecuta la siguiente celda después de correr todo el notebook para consolidar los hallazgos principales del estado inicial del dataset.

In [ ]:
summary = {
    "dataset_root": str(DATASET_ROOT),
    "total_images": len(image_inventory),
    "splits_detected": sorted(image_inventory["split"].unique().tolist()) if not image_inventory.empty else [],
    "image_formats": sorted(image_inventory["extension"].unique().tolist()) if not image_inventory.empty else [],
    "total_classes": len(class_names),
    "total_annotations": len(annotations_df),
    "invalid_or_unreadable_images": len(invalid_images_df),
    "color_modes": valid_images_df["mode"].value_counts().to_dict() if not valid_images_df.empty else {},
    "exact_duplicate_groups": int(duplicate_hashes["md5"].nunique()) if not duplicate_hashes.empty else 0,
    "images_in_exact_duplicate_groups": len(duplicate_hashes),
}

for key, value in summary.items():
    print(f"{key}: {value}")

### Conclusiones iniciales

- El dataset fue inspeccionado desde la ruta raíz descargada con `kagglehub`.
- Se revisó la estructura de carpetas, splits, formatos de imagen, tamaños de archivo, clases, anotaciones, dimensiones, modos de color, imágenes ilegibles, duplicados exactos y métricas básicas de calidad visual.
- Los posibles problemas detectados, como desbalance de clases, duplicados exactos, imágenes corruptas, tamaños atípicos, proporciones inusuales, bajo contraste, brillo extremo o desenfoque, deben revisarse posteriormente antes de entrenar modelos.
- No se realizó ninguna limpieza. No se eliminaron, movieron, redimensionaron, transformaron ni sobrescribieron imágenes o etiquetas.